In [ ]:
from statsbombpy import sb
import pandas as pd
import numpy as np


pd.set_option("display.max_columns", None)

import warnings

# turn off warnings for cleaner output
warnings.filterwarnings("ignore")

# retrieve 2003/2004 Premier League matches
matches = sb.matches(competition_id=2, season_id=44)

# get all the matches that Arsenal played
arsenal_matches = matches[
    (matches["home_team"] == "Arsenal") | (matches["away_team"] == "Arsenal")
]
# Get the first match Arsenal played in the dataset
match_id = arsenal_matches.iloc[0]["match_id"]
# match name for printing
match_name = (
    f"{arsenal_matches.iloc[0]['home_team']} vs {arsenal_matches.iloc[0]['away_team']}"
)
print(f"Fetching tick data for: {match_name}\n")
# get the data for every single event that occurred in the match: pass, tackle etc.
events = sb.events(match_id=match_id)
# break the [x, y] coordinates pair column into separate x and y columns
events["x"] = events["location"].apply(lambda x: x[0] if isinstance(x, list) else None)
events["y"] = events["location"].apply(lambda x: x[1] if isinstance(x, list) else None)

# add a column for how many seconds have passed in the period at the time of the event (details about period in dataset_info.md)
events["period_seconds"] = pd.to_timedelta(events["timestamp"]).dt.total_seconds()

# sort by period first, then by seconds passed in that period to ensure chronological order of events
# without reset_index, the row indices would be messed up because it would keep the unsorted values
# without drop=True, the old row indices would be kept as a new column. we don't need them
events = events.sort_values(by=["period", "period_seconds"]).reset_index(drop=True)

# drop any events where the match hadn't started yet
game_states = events.dropna(subset=["x", "y"]).copy()

# sort again by period and seconds in period
game_states = game_states.sort_values(by=["period", "period_seconds"]).reset_index(
    drop=True
)
# group by period, then find the time spent in each event
game_states["delta_t"] = game_states.groupby("period")["period_seconds"].diff()
# the first event will always have a NaN value becuase there was no previous event to compare the time to
game_states["delta_t"] = game_states["delta_t"].fillna(0)

print(game_states[["type", "period_seconds", "delta_t", "x", "y"]].head())

In [ ]:
print(game_states.head())

In [ ]:
events_35th_min = events[events["minute"] == 35].copy()

shots_df = events[events["type"] == "Shot"].copy()
print(shots_df[shots_df["shot_outcome"] == "Goal"])

loc_df = events.loc[:, "location"]

In [ ]:
import numpy as np

# create evenly spaced boundary lines for the x and y coordinates of pitch zones
x_edges = np.linspace(0, 120, 7)
y_edges = np.linspace(0, 80, 5)
# create new columns of x bin and y bins for zones of the pitch
game_states["x_bin"] = pd.cut(
    game_states["x"], bins=x_edges, labels=False, include_lowest=True
)
game_states["y_bin"] = pd.cut(
    game_states["y"], bins=y_edges, labels=False, include_lowest=True
)
# give each zone an id based on its x and y
game_states["zone_id"] = (game_states["x_bin"] * 4) + game_states["y_bin"]

print(game_states[["type", "x", "y", "x_bin", "y_bin", "zone_id"]].head(10))

In [ ]:
target_team = "Arsenal"
# create a new possession column with 1 for when the target team has possession and 0 otherwise
game_states["P"] = (game_states["possession_team"] == target_team).astype(int)

print(
    game_states[
        ["type", "minute", "second", "team", "possession_team", "P", "zone_id"]
    ].head(10)
)

In [ ]:
is_goal = (game_states["type"] == "Shot") & (game_states["shot_outcome"] == "Goal")

# if it's an arsenal goal, label it as 1 if both conditions are True, otherwise 0
game_states["arsenal_goal_event"] = np.where(
    is_goal & (game_states["team"] == target_team), 1, 0
)
game_states["opponent_goal_event"] = np.where(
    is_goal & (game_states["team"] != target_team), 1, 0
)
# arsenal's score is the sum of how many arsenal goal events there've been
game_states["arsenal_score"] = game_states["arsenal_goal_event"].cumsum()
game_states["opponent_score"] = game_states["opponent_goal_event"].cumsum()
# goal diff is diff in goals scored so far
game_states["delta_G"] = game_states["arsenal_score"] - game_states["opponent_score"]
# limit goal diff to between -2 and 2 because a team 3 up plays essentially identically to a team 5 up, but if we allow
# for teams to be 5 up there's not much data on that so the matrix will be sparse and therefore not good at predicting
game_states["delta_G"] = game_states["delta_G"].clip(-2, 2)

In [ ]:
# concatenate the data to create a state
game_states["state_id"] = (
    "G:"
    + game_states["delta_G"].astype(str)
    + "_P:"
    + game_states["P"].astype(str)
    + "_Z:"
    + game_states["zone_id"].astype(str)
)

print(
    game_states[
        ["minute", "second", "team", "type", "delta_G", "P", "zone_id", "state_id"]
    ].head(15)
)

In [ ]:
# shift(-1) means 'see the next row'
# so this essentially assigns the corresponding next_state_id for each state_id by creating a new column for the next one
game_states["next_state_id"] = game_states.groupby("period")["state_id"].shift(-1)

game_states["next_time"] = game_states.groupby("period")["period_seconds"].shift(-1)

# how long did the ball survive in a given state before moving to the next one
game_states["holding_time"] = game_states["next_time"] - game_states["period_seconds"]

# because the last event in a period won't have a next state, and so holding time is meaningless
transitions = game_states.dropna(subset=["next_state_id", "holding_time"]).copy()

print(transitions)

columns_to_view = [
    "period_seconds",
    "type",
    "state_id",
    "next_state_id",
    "holding_time",
]
# print(transitions[columns_to_view].head(10))

In [ ]:
# finds the number of times each exact state jump happened (e.g. how many times arsenal passed from the back to zone A4)
transition_counts = (
    transitions.groupby(["state_id", "next_state_id"]).size().reset_index(name="N_ij")
)
# finds the total time spent in each state
state_holding_times = (
    transitions.groupby("state_id")["holding_time"].sum().reset_index(name="T_i")
)
# aligns N_ij and T_i for each state_id so we're dividing the relevant numbers by each other
q_edges = pd.merge(transition_counts, state_holding_times, on="state_id")
q_edges["q_ij"] = q_edges["N_ij"] / q_edges["T_i"]

# takes the list of connections between states and their q_ij values and makes a 2x2 grid where the rows are the starting
# states, the columns are destination states, and each intersection of these has its q_ij value of average rate of transition
# I.e. if there were 100 states in total, then this would be a 100x100 grid
# state transitions that never happened are filled with 0
Q_matrix = q_edges.pivot(
    index="state_id", columns="next_state_id", values="q_ij"
).fillna(0.0)

# creates a list of all states by taking every state that appears as either a row or column
# a transition matrix needs to be square so we need to make sure every state appears, even if it doesn't have a matching
# state that it transitions out of (bc it's at the end of the match or something)
all_states = sorted(list(set(Q_matrix.index).union(set(Q_matrix.columns))))

# fills values that don't have a matching state to transition to with 0
Q_matrix = Q_matrix.reindex(index=all_states, columns=all_states, fill_value=0.0)

# sets the diagonals to 0 so they don't interact with the sum at all
np.fill_diagonal(Q_matrix.values, 0.0)
# finds the sums of the next state transition rates for each state
row_sums = Q_matrix.sum(axis=1)
# fills the diagonal with a number such that the rows will sum to 0
np.fill_diagonal(Q_matrix.values, -row_sums)

print(Q_matrix)

In [ ]:
from main import simulate_next_state

test_state = Q_matrix.index[10]
next_s, dt = simulate_next_state(test_state, Q_matrix)
print(f"Started in: {test_state}")
print(f"Jumped to: {next_s} after {dt:.2f} seconds")

In [ ]:
from main import simulate_match

test_outcome = simulate_match(
    initial_state="G:0_P:1_Z:12", Q=Q_matrix, current_minute=75, current_second=0
)
print(f"Simulated Outcome: {test_outcome}")

In [13]:
from main import run_monte_carlo

initial_test_state = "G:0_P:1_Z:14"
implied_odds = run_monte_carlo(
    initial_state=initial_test_state, Q=Q_matrix, current_minute=70, current_second=0
)
print(f"Arsenal Win: {implied_odds['Win_Prob']:.1%}")
print(f"Draw:        {implied_odds['Draw_Prob']:.1%}")
print(f"Arsenal Loss:{implied_odds['Loss_Prob']:.1%}")

KeyboardInterrupt: 